# Phase 1 — LLM Parsing & Content Extraction

**Reads:** `checkpoint/ocr_checkpoint.json` + `output/{classification}/{document_group_id}/{filename_stem}/page_{N}/`

**Writes:** `content_extraction/{classification}/{document_group_id}/{filename_stem}/`
```
  page_{N}/
    processed_content.txt   ← clean markdown (tables converted to row text)
    metadata.json           ← full metadata: images, tables, TOC info, CSV fields
  toc.json                  ← written at file level when TOC found
```

**Steps per page:**
1. Load MonkeyOCR markdown from Phase 1 OCR output
2. Detect TOC pages (first 6); optionally enable TOC navigation layer (80% rule)
3. Strip image tags → caption + path saved to metadata
4. Convert HTML tables → embed-format row text  *(or LLM parse — see toggle below)*
5. Clean markdown (dedup headers, repeating footers)
6. Write `processed_content.txt` + `metadata.json`
7. Checkpoint per page — safe to re-run after interruption

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 2. Table parsing mode — set this first

| Mode | What it does | GPU needed? |
|------|-------------|-------------|
| `"text"` | HTML → structured row text via BeautifulSoup (fast, no GPU) | No |
| `"llm"`  | HTML → FACT/ROW records via local Qwen model (slower) | Yes |

Set `TABLE_PARSE_MODE` then run all cells top to bottom.
GPU deps and model loading cells are **automatically skipped** when mode is `"text"`.

In [2]:
# ── TOGGLE ────────────────────────────────────────────────────────────────────
TABLE_PARSE_MODE = "text"   # "text"  or  "llm"
# ─────────────────────────────────────────────────────────────────────────────

assert TABLE_PARSE_MODE in ("text", "llm"),     f"TABLE_PARSE_MODE must be 'text' or 'llm', got: {TABLE_PARSE_MODE!r}"

print(f"Table parse mode : {TABLE_PARSE_MODE}")
if TABLE_PARSE_MODE == "text":
    print("  HTML tables -> embed-format row text (BeautifulSoup).")
    print("  GPU install and LLM loading cells will be SKIPPED.")
else:
    print("  HTML tables -> normalised by Qwen LLM.")
    print("  Run ALL cells including GPU install and LLM loading.")

Table parse mode : text
  HTML tables -> embed-format row text (BeautifulSoup).
  GPU install and LLM loading cells will be SKIPPED.


## 3. Install dependencies

In [3]:
# Always needed
!pip install beautifulsoup4 pandas tqdm -q

# Only needed for LLM mode
if TABLE_PARSE_MODE == "llm":
    !pip install transformers accelerate huggingface_hub safetensors -q

## 4. Copy helper scripts from Drive → VM

Upload these files to `RAG_OCRv1/scripts/` on your Drive:
- `general_processor.py`
- `general_content_extractor.py`
- `html_table_converter.py`
- `llm_table_normalizer.py`  *(only needed for llm mode)*

In [4]:
import shutil, sys
from pathlib import Path

DRIVE_BASE  = Path("/content/drive/MyDrive/RAG_OCRv1")
SCRIPTS_SRC = DRIVE_BASE / "scripts"
SCRIPTS_DST = Path("/content/rag_scripts")

SCRIPTS_DST.mkdir(exist_ok=True)
if str(SCRIPTS_DST) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DST))

always_scripts = [
    "general_processor.py",
    "general_content_extractor.py",
    "html_table_converter.py",
]
llm_scripts = ["llm_table_normalizer.py"]

scripts_to_copy = always_scripts + (llm_scripts if TABLE_PARSE_MODE == "llm" else [])

for script in scripts_to_copy:
    src = SCRIPTS_SRC / script
    dst = SCRIPTS_DST / script
    if src.exists():
        shutil.copy2(src, dst)
        print(f"  Copied : {script}")
    else:
        print(f"  MISSING: {src}  -- upload this file to Drive/scripts/")

print(f"sripts ready in {SCRIPTS_DST}")

  Copied : general_processor.py
  Copied : general_content_extractor.py
  Copied : html_table_converter.py
sripts ready in /content/rag_scripts


## 5. Configuration

In [5]:
from pathlib import Path

# ── Drive root ────────────────────────────────────────────────────────────────
DRIVE_BASE = Path("/content/drive/MyDrive/RAG_OCRv1")

# ── Input ─────────────────────────────────────────────────────────────────────
OCR_OUTPUT_DIR = DRIVE_BASE / "output"
CSV_PATH       = DRIVE_BASE / "data" / "data_information.csv"

# ── Output ────────────────────────────────────────────────────────────────────
EXTRACTION_DIR = DRIVE_BASE / "content_extraction"

# ── Checkpoints ───────────────────────────────────────────────────────────────
CHECKPOINT_DIR        = DRIVE_BASE / "checkpoint"
OCR_CHECKPOINT        = CHECKPOINT_DIR / "ocr_checkpoint.json"
EXTRACTION_CHECKPOINT = CHECKPOINT_DIR / "content_extraction_checkpoint.json"

# ── LLM config (only used when TABLE_PARSE_MODE = "llm") ─────────────────────
# LOAD_FROM_DRIVE = False means always download fresh from HuggingFace
LLM_MODEL_NAME     = "Qwen/Qwen3.5-4B"
LLM_CACHE_PATH     = Path("/content/model_cache")
LLM_MAX_NEW_TOKENS = 512

# ── Processing options ────────────────────────────────────────────────────────
TOC_NAV_THRESHOLD     = 0.80
DOC_GROUP_FILTER      = None   # e.g. "panasonic_aircon_CS-PW24KE"
CLASSIFICATION_FILTER = None   # e.g. "MANUAL"

# ── Create dirs ───────────────────────────────────────────────────────────────
EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
if TABLE_PARSE_MODE == "llm":
    LLM_CACHE_PATH.mkdir(parents=True, exist_ok=True)

print("Configuration set")
print(f"  Mode              : {TABLE_PARSE_MODE}")
print(f"  OCR output dir    : {OCR_OUTPUT_DIR}")
print(f"  Extraction dir    : {EXTRACTION_DIR}")
print(f"  OCR checkpoint    : {OCR_CHECKPOINT}")
print(f"  Extract checkpoint: {EXTRACTION_CHECKPOINT}")
print(f"  CSV               : {CSV_PATH}")

Configuration set
  Mode              : text
  OCR output dir    : /content/drive/MyDrive/RAG_OCRv1/output
  Extraction dir    : /content/drive/MyDrive/RAG_OCRv1/content_extraction
  OCR checkpoint    : /content/drive/MyDrive/RAG_OCRv1/checkpoint/ocr_checkpoint.json
  Extract checkpoint: /content/drive/MyDrive/RAG_OCRv1/checkpoint/content_extraction_checkpoint.json
  CSV               : /content/drive/MyDrive/RAG_OCRv1/data/data_information.csv


## 6. Load LLM  *(LLM mode only — skip if mode = "text")*

In [6]:
model     = None
tokenizer = None

if TABLE_PARSE_MODE == "llm":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    # Always download directly from HuggingFace (no Drive copy)
    print(f"Loading {LLM_MODEL_NAME} from HuggingFace -> {LLM_CACHE_PATH}...")
    print("(Downloads once; cached in LLM_CACHE_PATH for rest of session)")

    tokenizer = AutoTokenizer.from_pretrained(
        LLM_MODEL_NAME,
        cache_dir=str(LLM_CACHE_PATH),
        trust_remote_code=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        device_map="auto",
        cache_dir=str(LLM_CACHE_PATH),
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model.eval()
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"Model loaded on: {next(model.parameters()).device}")
    print("LLM ready")
else:
    print("Skipped -- TABLE_PARSE_MODE is 'text'")

Skipped -- TABLE_PARSE_MODE is 'text'


## 7. Load OCR checkpoint + CSV manifest

In [7]:
import json
import pandas as pd
from pathlib import Path

# ── OCR checkpoint ────────────────────────────────────────────────────────────
if not OCR_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"OCR checkpoint not found: {OCR_CHECKPOINT}\nRun phase1_ocr.ipynb first."
    )

with open(OCR_CHECKPOINT) as f:
    ocr_cp = json.load(f)

completed_files = [k for k, v in ocr_cp["files"].items() if v.get("status") == "completed"]
print(f"OCR checkpoint : {len(ocr_cp['files'])} total, {len(completed_files)} completed")

# ── CSV manifest ──────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df["classification"] = df["classification"].str.strip().str.upper()

# Lookup: (document_group_id, filename_stem) -> full row dict
csv_lookup = {}
for _, row in df.iterrows():
    stem = Path(str(row["filename"]).strip()).stem
    key  = (str(row["document_group_id"]).strip(), stem)
    csv_lookup[key] = {
        col: (str(row[col]).strip() if pd.notna(row.get(col)) else "")
        for col in df.columns
    }

print(f"CSV manifest   : {len(csv_lookup)} file entries")

OCR checkpoint : 2 total, 2 completed
CSV manifest   : 8 file entries


## 8. Extraction checkpoint helpers

In [8]:
import json
from datetime import datetime


def _load_extraction_checkpoint() -> dict:
    if EXTRACTION_CHECKPOINT.exists():
        try:
            with open(EXTRACTION_CHECKPOINT) as f:
                data = json.load(f)
            total = sum(len(v.get("completed_pages", {})) for v in data["files"].values())
            print(f"Extraction checkpoint: {len(data['files'])} file(s), {total} page(s) done")
            return data
        except Exception as e:
            print(f"Could not load checkpoint: {e} -- starting fresh")
    return {"last_updated": None, "files": {}}


def _save_extraction_checkpoint(data: dict):
    data["last_updated"] = datetime.now().isoformat()
    with open(EXTRACTION_CHECKPOINT, "w") as f:
        json.dump(data, f, indent=2)


def _ext_file_key(doc_group_id, classification, filename_stem):
    return f"{doc_group_id}|{classification}|{filename_stem}"


def _is_page_extracted(cp, fkey, page_num):
    return str(page_num) in cp["files"].get(fkey, {}).get("completed_pages", {})


def _mark_page_extracted(cp, fkey, page_num, out_dir):
    cp["files"].setdefault(fkey, {
        "status": "in_progress", "completed_pages": {},
        "failed_pages": {}, "started_at": datetime.now().isoformat(),
    })
    cp["files"][fkey]["completed_pages"][str(page_num)] = {
        "output_dir": str(out_dir), "timestamp": datetime.now().isoformat(),
    }
    _save_extraction_checkpoint(cp)


def _mark_page_extraction_failed(cp, fkey, page_num, error):
    cp["files"].setdefault(fkey, {
        "status": "in_progress", "completed_pages": {}, "failed_pages": {},
    })
    cp["files"][fkey]["failed_pages"][str(page_num)] = {
        "error": error, "timestamp": datetime.now().isoformat(),
    }
    _save_extraction_checkpoint(cp)


def _mark_file_extraction_complete(cp, fkey, total_pages, doc_group_id,
                                    filename_stem, classification, use_toc_nav):
    entry = cp["files"].get(fkey, {})
    entry.update({
        "status":            "completed",
        "total_pages":       total_pages,
        "completed_at":      datetime.now().isoformat(),
        "document_group_id": doc_group_id,
        "filename":          filename_stem,
        "classification":    classification,
        "use_toc_nav":       use_toc_nav,
        "table_parse_mode":  TABLE_PARSE_MODE,
    })
    cp["files"][fkey] = entry
    _save_extraction_checkpoint(cp)


ext_cp = _load_extraction_checkpoint()
print("Checkpoint helpers ready")

Checkpoint helpers ready


## 9. Build work list from OCR checkpoint

In [9]:
work_items = []

for ocr_key, ocr_entry in ocr_cp["files"].items():
    if ocr_entry.get("status") != "completed":
        continue

    doc_group_id, classification, filename_stem = ocr_key.split("|", 2)

    if DOC_GROUP_FILTER and doc_group_id != DOC_GROUP_FILTER:
        continue
    if CLASSIFICATION_FILTER and classification != CLASSIFICATION_FILTER.upper():
        continue

    file_ocr_dir = OCR_OUTPUT_DIR / classification / doc_group_id / filename_stem
    if not file_ocr_dir.exists():
        print(f"  OCR dir missing: {file_ocr_dir}")
        continue

    page_dirs = sorted(
        [d for d in file_ocr_dir.iterdir() if d.is_dir() and d.name.startswith("page_")],
        key=lambda d: int(d.name.split("_")[1]),
    )
    if not page_dirs:
        print(f"  No page dirs: {file_ocr_dir}")
        continue

    csv_meta = csv_lookup.get((doc_group_id, filename_stem), {})

    work_items.append({
        "doc_group_id":   doc_group_id,
        "classification": classification,
        "filename_stem":  filename_stem,
        "file_ocr_dir":   file_ocr_dir,
        "page_dirs":      page_dirs,
        "csv_meta":       csv_meta,
        "ocr_key":        ocr_key,
    })

print(f"Work list: {len(work_items)} file(s) ready for parsing")
for w in work_items:
    fkey   = _ext_file_key(w['doc_group_id'], w['classification'], w['filename_stem'])
    status = ext_cp["files"].get(fkey, {}).get("status", "not started")
    print(f"  [{w['classification']}] {w['doc_group_id']}/{w['filename_stem']} "
          f"| {len(w['page_dirs'])} pages | {status}")

Work list: 2 file(s) ready for parsing
  [MANUAL] panasonic_aircon_CS-PW24KE/CS-PW24KE_service_manual_PHAAM0810057C2 | 55 pages | not started
  [MANUAL] panasonic_aircon_CS-E12QD3EAW/CS-E12QD3EAW_SM_PAPAMY1406071CE | 95 pages | not started


## 10. Content Extraction

Table handling is determined by `TABLE_PARSE_MODE` set in Cell 2.
- **`text`** — BeautifulSoup → `[Row N] val | val |`
- **`llm`**  — Qwen → `FACT: key = value` / `ROW: v1 | v2`

In [10]:
from tqdm import tqdm
from general_processor import (
    detect_toc_pages,
    should_use_toc_nav,
    extract_images_from_markdown,
    clean_markdown,
    find_md_file,
)
from general_content_extractor import GeneralContentExtractor

# Bind the correct table parser based on mode
if TABLE_PARSE_MODE == "text":
    from html_table_converter import convert_tables_in_markdown
    def _parse_tables(md_text):
        return convert_tables_in_markdown(md_text)
else:
    from llm_table_normalizer import normalize_tables_in_markdown
    def _parse_tables(md_text):
        return normalize_tables_in_markdown(
            md_text, model, tokenizer, max_new_tokens=LLM_MAX_NEW_TOKENS
        )

print(f"Table parser: {TABLE_PARSE_MODE}")

# ── Summary counters ──────────────────────────────────────────────────────────
total_files_done   = 0
total_files_skip   = 0
total_pages_done   = 0
total_pages_skip   = 0
total_pages_failed = 0

print("=" * 70)
print(f"PHASE 1 - CONTENT EXTRACTION  [{TABLE_PARSE_MODE.upper()} mode]")
print("=" * 70)

file_bar = tqdm(work_items, unit="file", desc="Files", dynamic_ncols=True)

for work in file_bar:
    doc_group_id   = work["doc_group_id"]
    classification = work["classification"]
    filename_stem  = work["filename_stem"]
    page_dirs      = work["page_dirs"]
    csv_meta       = work["csv_meta"]

    fkey = _ext_file_key(doc_group_id, classification, filename_stem)
    file_bar.set_postfix({"doc": f"{doc_group_id[:18]}/{filename_stem[:16]}"})

    # Skip fully completed files
    if ext_cp["files"].get(fkey, {}).get("status") == "completed":
        tqdm.write(f"SKIP (done): {doc_group_id}/{filename_stem}")
        total_files_skip += 1
        continue

    tqdm.write(f"{'='*70}")
    tqdm.write(f"[{classification}] {doc_group_id} / {filename_stem}")
    tqdm.write(f"  Pages: {len(page_dirs)}  |  Table mode: {TABLE_PARSE_MODE}")

    # ── A: TOC detection ─────────────────────────────────────────────────
    toc_page_nums, toc_headings = detect_toc_pages(page_dirs, check_first_n=6)
    use_toc_nav    = False
    toc_page_texts = {}

    if toc_page_nums:
        tqdm.write(f"  TOC pages: {toc_page_nums} | {len(toc_headings)} heading(s)")
        for pg_num in toc_page_nums:
            pg_dir  = work["file_ocr_dir"] / f"page_{pg_num}"
            md_file = find_md_file(pg_dir)
            if md_file:
                try:
                    toc_page_texts[pg_num] = md_file.read_text(encoding="utf-8")
                except Exception:
                    pass
        use_toc_nav = should_use_toc_nav(
            toc_headings, page_dirs, threshold=TOC_NAV_THRESHOLD
        )
    else:
        tqdm.write("  No TOC detected")

    # ── B: Per-page processing ────────────────────────────────────────────
    extractor = GeneralContentExtractor(
        output_root=EXTRACTION_DIR,
        csv_metadata=csv_meta,
    )

    file_pages_done = file_pages_skip = file_pages_failed = 0

    page_bar = tqdm(
        page_dirs, unit="pg",
        desc=f"  [{filename_stem[:20]}]",
        dynamic_ncols=True, leave=False,
    )

    for page_dir in page_bar:
        page_num = int(page_dir.name.split("_")[1])

        # TOC pages go to toc.json, not content units
        if page_num in toc_page_nums:
            page_bar.set_postfix({"p": page_num, "status": "TOC->skip"})
            file_pages_skip  += 1
            total_pages_skip += 1
            continue

        # Skip already-done pages
        if _is_page_extracted(ext_cp, fkey, page_num):
            page_bar.set_postfix({"p": page_num, "status": "cached"})
            file_pages_skip  += 1
            total_pages_skip += 1
            continue

        try:
            # 1. Load OCR markdown
            page_bar.set_postfix({"p": page_num, "status": "load"})
            md_file = find_md_file(page_dir)
            if md_file is None:
                raise FileNotFoundError(f"No markdown in {page_dir}")
            raw_md = md_file.read_text(encoding="utf-8")

            # 2. Strip image tags
            page_bar.set_postfix({"p": page_num, "status": "images"})
            md_no_imgs, image_records = extract_images_from_markdown(raw_md, page_dir)

            # 3. Table conversion (text or llm)
            page_bar.set_postfix({"p": page_num, "status": f"tables({TABLE_PARSE_MODE})"})
            md_no_tables, table_records = _parse_tables(md_no_imgs)

            # 4. Clean markdown
            page_bar.set_postfix({"p": page_num, "status": "clean"})
            cleaned_md = clean_markdown(md_no_tables)

            # 5. Write output
            page_bar.set_postfix({"p": page_num, "status": "write"})
            unit = extractor.process_page(
                document_group_id=doc_group_id,
                filename_stem=filename_stem,
                classification=classification,
                page_num=page_num,
                ocr_page_dir=page_dir,
                cleaned_md=cleaned_md,
                image_records=image_records,
                table_records=table_records,
                toc_page_nums=toc_page_nums,
                use_toc_nav=use_toc_nav,
                toc_headings=toc_headings,
            )

            out_dir = (EXTRACTION_DIR / classification / doc_group_id
                       / filename_stem / f"page_{page_num}")
            _mark_page_extracted(ext_cp, fkey, page_num, out_dir)
            page_bar.set_postfix({"p": page_num, "status": "done"})
            file_pages_done  += 1
            total_pages_done += 1

        except Exception as e:
            err = str(e)[:300]
            tqdm.write(f"    p{page_num} FAILED: {err}")
            _mark_page_extraction_failed(ext_cp, fkey, page_num, err)
            file_pages_failed  += 1
            total_pages_failed += 1

    page_bar.close()

    # ── C: toc.json ───────────────────────────────────────────────────────
    if toc_page_nums:
        extractor.write_toc_json(
            document_group_id=doc_group_id,
            filename_stem=filename_stem,
            classification=classification,
            toc_page_nums=toc_page_nums,
            toc_headings=toc_headings,
            use_toc_nav=use_toc_nav,
            toc_page_texts=toc_page_texts,
        )

    # ── D: Finalise ───────────────────────────────────────────────────────
    done_count   = len(ext_cp["files"].get(fkey, {}).get("completed_pages", {}))
    failed_count = len(ext_cp["files"].get(fkey, {}).get("failed_pages", {}))
    body_pages   = len(page_dirs) - len(toc_page_nums)

    if failed_count == 0:
        _mark_file_extraction_complete(
            ext_cp, fkey, body_pages, doc_group_id,
            filename_stem, classification, use_toc_nav
        )
        tqdm.write(f"  File complete: {done_count}/{body_pages} body pages")
        total_files_done += 1
    else:
        _save_extraction_checkpoint(ext_cp)
        tqdm.write(f"  Partial: {done_count} done, {failed_count} failed / {body_pages} body pages")

    tqdm.write(
        f"  Pages -- done: {file_pages_done}  "
        f"skipped: {file_pages_skip}  failed: {file_pages_failed}"
    )

file_bar.close()

print("" + "=" * 70)
print("PHASE 1 CONTENT EXTRACTION - COMPLETE")
print("=" * 70)
print(f"  Mode             : {TABLE_PARSE_MODE}")
print(f"  Files completed  : {total_files_done}")
print(f"  Files skipped    : {total_files_skip}")
print(f"  Pages done       : {total_pages_done}")
print(f"  Pages skipped    : {total_pages_skip}")
print(f"  Pages failed     : {total_pages_failed}")
print(f"  Checkpoint       : {EXTRACTION_CHECKPOINT}")
print("=" * 70)

Table parser: text
PHASE 1 - CONTENT EXTRACTION  [TEXT mode]


Files:   0%|          | 0/2 [00:00<?, ?file/s, doc=panasonic_aircon_C/CS-PW24KE_servic]

[MANUAL] panasonic_aircon_CS-PW24KE / CS-PW24KE_service_manual_PHAAM0810057C2
  Pages: 55  |  Table mode: text


Files:   0%|          | 0/2 [00:03<?, ?file/s, doc=panasonic_aircon_C/CS-PW24KE_servic]

  TOC pages: [1, 2, 3, 4] | 39 heading(s)
   TOC nav check: 35/39 headings matched (90%) — threshold 80%


...

    page   88:    433 chars | 3 img | 0 tbl | → page_88/
    page   89:    280 chars | 3 img | 0 tbl | → page_89/
    page   90:    284 chars | 2 img | 0 tbl | → page_90/



  [CS-E12QD3EAW_SM_PAPA]:  96%|█████████▌| 91/95 [00:06<00:00, 13.76pg/s, p=92, status=clean]       

  [CS-E12QD3EAW_SM_PAPA]:  96%|█████████▌| 91/95 [00:06<00:00, 13.76pg/s, p=93, status=clean]       

  [CS-E12QD3EAW_SM_PAPA]:  98%|█████████▊| 93/95 [00:06<00:00, 12.93pg/s, p=94, status=load] 


    page   91:    233 chars | 1 img | 0 tbl | → page_91/
    p92 FAILED: 'normalised'
    p93 FAILED: 'normalised'


  [CS-E12QD3EAW_SM_PAPA]:  98%|█████████▊| 93/95 [00:06<00:00, 12.93pg/s, p=95, status=clean]       

Files: 100%|██████████| 2/2 [01:03<00:00, 31.84s/file, doc=panasonic_aircon_C/CS-E12QD3EAW_SM_]

    page   94:    170 chars | 1 img | 0 tbl | → page_94/
    p95 FAILED: 'normalised'
   TOC saved: 56 headings | nav_layer=ENABLED → toc.json
  Partial: 59 done, 34 failed / 93 body pages
  Pages -- done: 59  skipped: 2  failed: 34
PHASE 1 CONTENT EXTRACTION - COMPLETE
  Mode             : text
  Files completed  : 0
  Files skipped    : 0
  Pages done       : 91
  Pages skipped    : 6
  Pages failed     : 53
  Checkpoint       : /content/drive/MyDrive/RAG_OCRv1/checkpoint/content_extraction_checkpoint.json


## 11. Inspection (optional)

In [ ]:
import json

if EXTRACTION_CHECKPOINT.exists():
    cp = json.loads(EXTRACTION_CHECKPOINT.read_text())
    print(f"Last updated : {cp.get('last_updated')}")
    print(f"Total files  : {len(cp['files'])}")
    for key, entry in cp["files"].items():
        doc_group_id, classification, filename_stem = key.split("|", 2)
        n_done   = len(entry.get("completed_pages", {}))
        n_failed = len(entry.get("failed_pages", {}))
        total    = entry.get("total_pages", "?")
        status   = entry.get("status", "unknown")
        mode     = entry.get("table_parse_mode", "?")
        toc_nav  = entry.get("use_toc_nav", None)
        icon     = "OK" if status == "completed" else ("FAIL" if n_failed else "...")
        toc_str  = "TOC-NAV" if toc_nav else ("toc-stored" if toc_nav is not None else "no-toc")
        print(f"  [{icon}] [{classification}] {doc_group_id}/{filename_stem}")
        print(f"         {n_done}/{total} pages | {n_failed} failed | mode={mode} | {toc_str}")
        if n_failed:
            for pg, info in entry["failed_pages"].items():
                print(f"p{pg}: {info['error'][:100]}")
else:
    print("No extraction checkpoint found yet.")

## 12. Inspect a single page (optional)

In [ ]:
import json

INSPECT_DOC_GROUP = None   # e.g. "panasonic_aircon_CS-PW24KE"
INSPECT_FILENAME  = None   # e.g. "CS-PW24KE_service_manual_PHAAM0810057C2"
INSPECT_CLASSIF   = None   # e.g. "MANUAL"
INSPECT_PAGE      = 1

if INSPECT_DOC_GROUP and INSPECT_FILENAME and INSPECT_CLASSIF:
    page_dir = (EXTRACTION_DIR / INSPECT_CLASSIF / INSPECT_DOC_GROUP
                / INSPECT_FILENAME / f"page_{INSPECT_PAGE}")
    txt  = page_dir / "processed_content.txt"
    meta = page_dir / "metadata.json"

    if txt.exists():
        print("=" * 60)
        print("processed_content.txt (first 60 lines):")
        print("=" * 60)
        print("".join(txt.read_text(encoding="utf-8").splitlines()[:60]))

    if meta.exists():
        data = json.loads(meta.read_text())
        print("" + "=" * 60)
        print("metadata.json (key fields):")
        print("=" * 60)
        for k in ["unit_id", "page", "classification", "text_length",
                  "total_tables", "total_figures", "use_toc_nav",
                  "toc_page_nums", "model_number", "category_level_1",
                  "date_added", "notes"]:
            if k in data:
                print(f"  {k}: {data[k]}")
        if data.get("images"):
            print(f"  images ({len(data['images'])}): {data['images'][:2]}")
        if data.get("tables"):
            t   = data["tables"][0]
            key = "embed_text" if "embed_text" in t else "normalised"
            print(f"  tables ({len(data['tables'])}) first preview:")
            print(f"    {t[key][:300]}")
else:
    print("Set INSPECT_* variables above to inspect a specific page.")

## 13. Clear GPU memory (optional — LLM mode only)

In [ ]:
if TABLE_PARSE_MODE == "llm" and model is not None:
    # import torch, gc
    # del model
    # del tokenizer
    # gc.collect()
    # torch.cuda.empty_cache()
    print("GPU memory cleared")
else:
    print("Nothing to clear.")